In [1]:
import pandas as pd
import os

In [2]:
path = "/Users/sanjivjha/Developer/ubicomp-capstone/pacakging-scenario-data/out.csv"
df =  pd.read_csv(path)
df.info

<bound method DataFrame.info of           accel_x      accel_y      accel_z       gyro_x       gyro_y  \
0     3873.259080 -7771.304469  -110.936223 -7298.429725 -4186.429420   
1     1597.283767 -4154.439848   -58.442177 -6375.378310 -5065.628894   
2     1458.218902 -4102.852987  -258.115603 -6258.237004 -4977.078758   
3     1253.527259 -2480.549899  -201.001613 -6122.498678 -4556.003768   
4        3.628252  1494.272413    48.251777 -5817.556721 -4504.386682   
...           ...          ...          ...          ...          ...   
3107  -359.856993 -3920.658676   874.414117 -2969.624428 -7377.954120   
3108 -1483.607960  -209.781789  1576.497752 -3241.207898 -7145.491663   
3109 -1550.197051  2588.640831  3470.563714 -2789.401435 -7369.527062   
3110 -1372.987614  3130.550171  4066.107840 -3160.854257 -7103.494737   
3111 -3004.544677    21.489666  1412.048956 -5842.671851 -5527.445230   

           gyro_z    temp     timestamp sender  
0     3061.570252  29.000  1.747745e+09   

In [3]:
# Step 1: Drop unnecessary columns
#df = df.drop(columns=['temp'])

# Step 2: Split into leg and hand data
leg = df[df['sender'] == 'leg'].drop(columns=['sender']).sort_values('timestamp')
hand = df[df['sender'] == 'hand'].drop(columns=['sender']).sort_values('timestamp')

# Step 3: Rename columns to differentiate
leg = leg.add_prefix('leg_')
hand = hand.add_prefix('hand_')

# Step 4: Rename timestamp columns back to match for merging
leg = leg.rename(columns={'leg_timestamp': 'timestamp'})
hand = hand.rename(columns={'hand_timestamp': 'timestamp'})

# Step 5: Merge on timestamp
merged = pd.merge_asof(hand, leg, on='timestamp', direction='nearest', tolerance=2)

# Step 6: Drop unnecessary columns
#merged = merged.drop(columns=['timestamp', 'timestamp'])
duplicates = merged.columns.duplicated()

# Keep only the first occurrence of each column name
merged = merged.loc[:, ~duplicates]

#drop one of the temperature columns
merged = merged.drop(columns=['leg_temp'])

# Step 7: Rename columns to match IWMSHA
merged = merged.rename(columns={'leg_gyro_x':'gx2', 'leg_gyro_y':'gy2', 'leg_gyro_z':'gz2', 'leg_accel_x':'ax2', 'leg_accel_y':'ay2', 'leg_accel_z':'az2',   'hand_gyro_x':'gx1',  'hand_gyro_y':'gy1',  'hand_gyro_z':'gz1', 'hand_accel_x':'ax1',  'hand_accel_y':'ay1',  'hand_accel_z':'az1', "hand_temp":'temperature' })

print(merged.info)

<bound method DataFrame.info of               ax1           ay1          az1          gx1          gy1  \
0     3873.259080  -7771.304469  -110.936223 -7298.429725 -4186.429420   
1     1458.218902  -4102.852987  -258.115603 -6258.237004 -4977.078758   
2        3.628252   1494.272413    48.251777 -5817.556721 -4504.386682   
3      -91.962133   1518.255234  -359.360502 -6764.190889 -4490.952219   
4     -901.329805   3803.195780   236.961450 -6805.622784 -4632.670799   
...           ...           ...          ...          ...          ...   
1552 -1454.113995  -7796.948050   674.369930 -4403.018585 -7040.819928   
1553 -3048.869823 -10454.408938  1411.961847 -3979.573397 -7353.332266   
1554   -37.838449  -7677.233682   969.242659 -2802.614210 -7688.899711   
1555 -1483.607960   -209.781789  1576.497752 -3241.207898 -7145.491663   
1556 -1372.987614   3130.550171  4066.107840 -3160.854257 -7103.494737   

              gz1  temperature     timestamp          ax2          ay2  \
0    

In [4]:
if os.path.exists(path):    
    os.renames(path, f"{path}.old")

merged.to_csv(path)

merged =  pd.read_csv(path)
merged.head()

,Unnamed: 0,ax1,ay1,az1,gx1,gy1,gz1,temperature,timestamp,ax2,ay2,az2,gx2,gy2,gz2
0,0,3873.259080,-7771.304469,-110.936223,-7298.429725,-4186.429420,3061.570252,29.000,1.747745e+09,1597.283767,-4154.439848,-58.442177,-6375.378310,-5065.628894,238.459112
1,1,1458.218902,-4102.852987,-258.115603,-6258.237004,-4977.078758,148.864649,29.125,1.747745e+09,1597.283767,-4154.439848,-58.442177,-6375.378310,-5065.628894,238.459112
2,2,3.628252,1494.272413,48.251777,-5817.556721,-4504.386682,283.899428,29.125,1.747745e+09,1253.527259,-2480.549899,-201.001613,-6122.498678,-4556.003768,763.558774
3,3,-91.962133,1518.255234,-359.360502,-6764.190889,-4490.952219,316.175018,29.125,1.747745e+09,-433.659858,2578.873679,-89.909438,-6932.625321,-4544.930274,490.449966
4,4,-901.329805,3803.195780,236.961450,-6805.622784,-4632.670799,522.084996,29.125,1.747745e+09,-433.659858,2578.873679,-89.909438,-6932.625321,-4544.930274,490.449966
